In [9]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *
from spark_session import get_spark

spark = get_spark("FastFood-P3-DataFrames")
sc = spark.sparkContext

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("restaurant_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("city", StringType()),
    StructField("status", StringType()),
    StructField("amount_mad", DoubleType()),
    StructField("items_count", IntegerType()),
    StructField("distance_km", DoubleType()),
    StructField("courier_id", StringType()),
    StructField("event_time", TimestampType()),
])
restaurants_schema = StructType([
    StructField("restaurant_id", StringType(), False),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("category", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
])

ORDERS = "s3a://lake/bronze/history/orders_history.jsonl"
RESTOS = "s3a://lake/bronze/referentiel/restaurants.json"

orders = (spark.read.schema(orders_schema)
          .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssX")
          .json(ORDERS))
restaurants = spark.read.schema(restaurants_schema).json(RESTOS)

orders.printSchema()
orders.show(3, truncate=False)
print(orders.count(), restaurants.count())

root
 |-- order_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- items_count: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- courier_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



+-----------+-------------+-----------+----------+-------+----------+-----------+-----------+----------+-------------------+
|order_id   |restaurant_id|customer_id|city      |status |amount_mad|items_count|distance_km|courier_id|event_time         |
+-----------+-------------+-----------+----------+-------+----------+-----------+-----------+----------+-------------------+
|ORD-0000001|R-088        |C-01101    |Rabat     |CREATED|189.05    |3          |2.4        |L-RAB-08  |2026-11-06 16:53:50|
|ORD-0000002|R-097        |C-00018    |Rabat     |CREATED|134.67    |2          |8.6        |L-RAB-07  |2026-11-06 16:54:02|
|ORD-0000003|R-179        |C-04530    |Casablanca|CREATED|323.43    |6          |3.2        |L-CAS-12  |2026-11-06 16:54:14|
+-----------+-------------+-----------+----------+-------+----------+-----------+-----------+----------+-------------------+
only showing top 3 rows



50003 200


In [10]:
clean = orders.dropDuplicates(["order_id", "status"])
delivered = clean.filter(F.col("status") == "DELIVERED")

ca_df = (delivered.groupBy("city")
         .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
              F.count("*").alias("nb"),
              F.round(F.avg("amount_mad"), 2).alias("panier_moyen"))
         .orderBy(F.desc("ca_mad")))
ca_df.show()

clean.createOrReplaceTempView("orders")
spark.sql("""
    SELECT city,
           ROUND(SUM(amount_mad), 2) AS ca_mad,
           COUNT(*)                  AS nb,
           ROUND(AVG(amount_mad), 2) AS panier_moyen
    FROM orders
    WHERE status = 'DELIVERED'
    GROUP BY city
    ORDER BY ca_mad DESC
""").show()

+----------+----------+----+------------+
|      city|    ca_mad|  nb|panier_moyen|
+----------+----------+----+------------+
|Casablanca|1405702.07|6010|      233.89|
|     Rabat| 832840.93|3726|      223.52|
| Marrakech| 673953.99|3026|      222.72|
|    Tanger| 493889.59|2206|      223.88|
+----------+----------+----+------------+



[Stage 13:>                                                         (0 + 3) / 3]

+----------+----------+----+------------+
|      city|    ca_mad|  nb|panier_moyen|
+----------+----------+----+------------+
|Casablanca|1405702.07|6010|      233.89|
|     Rabat| 832840.93|3726|      223.52|
| Marrakech| 673953.99|3026|      222.72|
|    Tanger| 493889.59|2206|      223.88|
+----------+----------+----+------------+



In [11]:
def ca_par_ville_df():
    return (spark.read.schema(orders_schema).json(ORDERS)
            .dropDuplicates(["order_id", "status"])
            .filter(F.col("status") == "DELIVERED")
            .groupBy("city").agg(F.sum("amount_mad"))
            .collect())

ca_par_ville_df()                         # chauffe
temps = []
for _ in range(3):
    t = time.time(); ca_par_ville_df(); temps.append(time.time() - t)
print("Temps DataFrame (s) :", [round(x, 2) for x in temps], "→ médiane", round(sorted(temps)[1], 2))

Temps DataFrame (s) : [1.44, 0.86, 0.86] → médiane 0.86


In [12]:
joined = delivered.join(restaurants.select("restaurant_id", "category", "name"),
                        on="restaurant_id", how="left")

ca_cat_ville = (joined.groupBy("city", "category")
                .agg(F.round(F.sum("amount_mad"), 2).alias("ca_mad"),
                     F.count("*").alias("nb"))
                .orderBy("city", F.desc("ca_mad")))
ca_cat_ville.show(40)

+----------+---------+---------+----+
|      city| category|   ca_mad|  nb|
+----------+---------+---------+----+
|Casablanca|  healthy|364316.39|1559|
|Casablanca|asiatique|262036.01| 938|
|Casablanca|    sushi|256145.69| 804|
|Casablanca|   burger|145323.04| 740|
|Casablanca|    tacos|106037.24| 645|
|Casablanca| marocain|100550.69| 492|
|Casablanca|    pizza| 89686.61| 368|
|Casablanca|   poulet|  81606.4| 464|
| Marrakech|    sushi|134569.01| 430|
| Marrakech|asiatique|127761.36| 486|
| Marrakech|    tacos| 121169.0| 721|
| Marrakech|   burger|101461.35| 512|
| Marrakech|    pizza| 56250.04| 236|
| Marrakech|  healthy| 52626.29| 228|
| Marrakech| marocain|  47705.5| 231|
| Marrakech|   poulet| 32411.44| 182|
|     Rabat|    pizza|172685.61| 747|
|     Rabat|   burger|172076.72| 888|
|     Rabat|  healthy|164224.87| 732|
|     Rabat|asiatique|121836.98| 461|
|     Rabat| marocain| 88366.76| 406|
|     Rabat|    sushi|  61745.3| 202|
|     Rabat|   poulet|  33805.9| 180|
|     Rabat|

In [13]:
from pyspark.sql.functions import udf

@udf(returnType=StringType())
def distance_band(d):
    if d is None:
        return None
    if d < 2:
        return "court"
    if d <= 5:
        return "moyen"
    return "long"

band_native = (F.when(F.col("distance_km") < 2, "court")
                .when(F.col("distance_km") <= 5, "moyen")
                .otherwise("long"))

def bench(col_expr, label, runs=3):
    # groupBy force l'évaluation de la colonne (un simple count() la ferait élaguer par Catalyst)
    f = lambda: clean.withColumn("distance_band", col_expr).groupBy("distance_band").count().collect()
    f()
    t = []
    for _ in range(runs):
        s = time.time(); f(); t.append(time.time() - s)
    print(f"{label:<18} médiane {sorted(t)[1]:.2f} s")

clean.cache().count()
bench(distance_band(F.col("distance_km")), "UDF Python")
bench(band_native, "when/otherwise")

clean.withColumn("distance_band", band_native).groupBy("distance_band").count().show()
clean.withColumn("distance_band", distance_band("distance_km")).select("distance_band").explain()
clean.withColumn("distance_band", band_native).select("distance_band").explain()

UDF Python         médiane 0.43 s
when/otherwise     médiane 0.25 s
+-------------+-----+
|distance_band|count|
+-------------+-----+
|         long| 6216|
|        court|13565|
|        moyen|29210|
+-------------+-----+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [pythonUDF0#8306 AS distance_band#8193]
   +- BatchEvalPython [distance_band(distance_km#4518)#8192], [pythonUDF0#8306]
      +- InMemoryTableScan [distance_km#4518]
            +- InMemoryRelation [order_id#4511, restaurant_id#4512, customer_id#4513, city#4514, status#4515, amount_mad#4516, items_count#4517, distance_km#4518, courier_id#4519, event_time#4520], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- AdaptiveSparkPlan isFinalPlan=false
                     +- SortAggregate(key=[order_id#4511, status#4515], functions=[first(restaurant_id#4512, false), first(customer_id#4513, false), first(city#4514, false), first(amount_mad#4516, false), first(items_count#4517, false),

### UDF Python ou expression native
| Version | Temps médian |
|---|---|
| UDF `distance_band` | … s |
| `when/otherwise` | … s |

Le plan physique de la version UDF contient un nœud **`BatchEvalPython`**. Pour chaque lot de lignes, Spark doit :
1. sérialiser les valeurs depuis la JVM (format Tungsten) vers un processus Python, via des sockets et Pickle ou Arrow ;
2. exécuter la fonction Python ligne par ligne ;
3. renvoyer et désérialiser les résultats vers la JVM.

L'UDF est en plus une **boîte noire pour Catalyst** : il ne peut ni l'analyser, ni la simplifier, ni la combiner avec d'autres expressions, ni la pousser vers la source.

La version `when/otherwise` est traduite en une expression **`CASE WHEN`** native, dans un simple `Project`. Catalyst la connaît et peut l'optimiser, et Tungsten la compile en bytecode JVM par *whole-stage code generation*. Elle s'exécute dans l'executor, sans aller-retour vers Python ni sérialisation, d'où le meilleur temps. Règle à retenir : n'utiliser une UDF Python qu'en dernier recours, quand aucune fonction de `pyspark.sql.functions` ne fait le travail.

On note aussi dans les deux plans l'`InMemoryRelation`, qui correspond au `cache()` de `clean`, et l'`Exchange hashpartitioning(order_id, status, 200)` : c'est le shuffle du `dropDuplicates`.

In [14]:
silver = (clean
          .join(restaurants.select("restaurant_id", "name", "category"), "restaurant_id", "left")
          .withColumn("distance_band", band_native)
          .withColumn("date", F.to_date("event_time")))

(silver.repartition("city").write.mode("overwrite")
       .partitionBy("city")
       .parquet("s3a://lake/silver/orders_history/"))

spark.read.parquet("s3a://lake/silver/orders_history/").groupBy("city").count().show()

+----------+-----+
|      city|count|
+----------+-----+
|     Rabat|12186|
|    Tanger| 7265|
|Casablanca|19622|
| Marrakech| 9918|
+----------+-----+



In [15]:
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [restaurant_id])
:- Filter (status#4515 = DELIVERED)
:  +- Deduplicate [order_id#4511, status#4515]
:     +- Relation [order_id#4511,restaurant_id#4512,customer_id#4513,city#4514,status#4515,amount_mad#4516,items_count#4517,distance_km#4518,courier_id#4519,event_time#4520] json
+- Project [restaurant_id#4531, category#4534, name#4532]
   +- Relation [restaurant_id#4531,name#4532,city#4533,category#4534,latitude#4535,longitude#4536] json

== Analyzed Logical Plan ==
restaurant_id: string, order_id: string, customer_id: string, city: string, status: string, amount_mad: double, items_count: int, distance_km: double, courier_id: string, event_time: timestamp, category: string, name: string
Project [restaurant_id#4512, order_id#4511, customer_id#4513, city#4514, status#4515, amount_mad#4516, items_count#4517, distance_km#4518, courier_id#4519, event_time#4520, category#4534, name#4532]
+- Join LeftOuter, (restaurant_id#4512 = restaurant_i

### Lecture du plan de la jointure

**1. Parsed Logical Plan** : la traduction brute du code, pas encore vérifiée. `'Join UsingJoin(LeftOuter, [restaurant_id])` porte une apostrophe, qui signale que les colonnes ne sont pas encore résolues.

**2. Analyzed Logical Plan** : l'Analyzer résout les noms et les types grâce au catalogue et aux schémas explicites. Chaque colonne reçoit un identifiant unique (`restaurant_id#1` côté commandes, `restaurant_id#20` côté référentiel). La clause `USING` devient la condition `restaurant_id#1 = restaurant_id#20`, et le schéma de sortie est entièrement typé.

**3. Optimized Logical Plan** : les règles de Catalyst s'appliquent.
- **Inférence de prédicats** : ajout de `isnotnull(status)` et `isnotnull(restaurant_id)`. Une valeur nulle ne peut pas satisfaire une égalité, donc on l'élimine au plus tôt.
- **Élagage de colonnes** : côté référentiel, seules `restaurant_id`, `name` et `category` sont gardées. `city`, `latitude` et `longitude` ne seront jamais lues.
- **Réutilisation du cache** : la branche commandes est remplacée par l'`InMemoryRelation` de `clean`, donc pas de relecture du JSON.

**4. Physical Plan** : Catalyst choisit la stratégie d'exécution.
- **`BroadcastHashJoin … LeftOuter, BuildRight`** : le référentiel (200 lignes, environ 29 Ko) est sous le seuil `spark.sql.autoBroadcastJoinThreshold` (10 Mo). Il est collecté puis **diffusé à chaque executor** (`BroadcastExchange`) sous forme de table de hachage. Chaque partition des commandes fait la jointure localement, **sans shuffle de la grande table**. Un `SortMergeJoin` aurait demandé de redistribuer et trier les deux côtés sur `restaurant_id`.
- **Pushdown vers la source** : le `FileScan json` du référentiel affiche `ReadSchema: struct<restaurant_id, name, category>` et `PushedFilters: [IsNotNull(restaurant_id)]`. La projection et le filtre sont appliqués dès la lecture.
- Le filtre `status = DELIVERED` est appliqué sur le cache (`InMemoryTableScan … [(status = DELIVERED)]`).
- Le seul `Exchange hashpartitioning(order_id, status, 200)` restant est celui du `dropDuplicates`, pas celui de la jointure.
- `AdaptiveSparkPlan isFinalPlan=false` : l'**AQE** pourra encore ajuster le plan à l'exécution, par exemple en fusionnant les petites partitions de shuffle.

In [8]:
spark.stop()

### Plan exécuté de la requête « CA par ville » (Spark UI, Query 32)
- **Scan json : 15 296 lignes en sortie** (sur 50 003). Catalyst a **poussé le filtre `status = 'DELIVERED'` sous le dédoublonnage** : le filtre porte sur une colonne de la clé de `dropDuplicates`, donc filtrer avant ou après donne le même résultat. On dédoublonne ainsi 3 fois moins de lignes.
- **SortAggregate + Exchange (14 968 lignes)** : c'est le `dropDuplicates(order_id, status)`, avec 15 296 → 14 968, soit 328 doublons DELIVERED retirés. L'Exchange est le shuffle sur la clé de dédoublonnage (environ 283 Ko).
- **HashAggregate (partial) → 4 lignes → Exchange → HashAggregate (final)** : c'est l'agrégation en deux phases, l'équivalent du combiner de `reduceByKey`. Seulement **4 enregistrements (270 octets)** traversent le second shuffle.
- **WholeStageCodegen (1) à (4)** : Tungsten fusionne les opérateurs de chaque bloc en une seule fonction Java compilée.
- **AQEShuffleRead, « number of coalesced partitions: 1 »** : l'AQE a mesuré la taille réelle des données (environ 300 Ko) et fusionné les 8 partitions de shuffle en 1, ce qui évite de lancer des tâches presque vides.

### Plan exécuté de la jointure (Spark UI, Query 33)
- **Branche gauche** (commandes) : Scan json (15 296 lignes) → Filter → dédoublonnage (SortAggregate + Exchange) → 14 968 lignes.
- **Branche droite** (référentiel) : Scan json (200 lignes, 28,5 Ko) → Filter `isnotnull` → **BroadcastExchange** : le driver collecte les 200 lignes (275 ms), construit une table de hachage (environ 1 Mo) et l'envoie à chaque executor.
- **BroadcastHashJoin → 14 968 lignes** : chaque partition des commandes fait la jointure **localement**, dans le même bloc WholeStageCodegen (4). La grande table **n'est pas shufflée** pour la jointure.
- **HashAggregate partial → 32 lignes** (4 villes × 8 catégories) → Exchange de **1,7 Ko** seulement → HashAggregate final → `TakeOrderedAndProject` (le `orderBy` + `show`).